# EDGAR Continued Pretraining — Part 1

Adapted from **Assignment 2, Part 1** (continued pretraining of GPT-2 + perplexity comparison).

**What changed vs. the class version:** the domain corpus is swapped from CMU Movie Summaries to **EDGAR 10-K filing text** (`eloukas/edgar-corpus`). Everything else — the GPT-2 model, the perplexity machinery, the training loop — mirrors what you did in class.

**The lesson to reproduce:** continued pretraining on financial text should *lower* perplexity on in-domain (EDGAR) text and *raise* it on out-of-domain (general / Wikipedia) text. That trade-off is the core intuition behind why a domain model like **FinBERT** beats vanilla BERT on financial tasks.

> **Run this in Google Colab with a T4 GPU** (Runtime → Change runtime type → T4 GPU). It needs a GPU.

## 0. Setup

In [ ]:
# Colab setup. We pin `datasets` to the class version because edgar-corpus and
# wikitext are script-based datasets that newer versions of `datasets` no longer load.
!pip install -q -U transformers
!pip install -q datasets==2.21.0

In [ ]:
import copy, random, itertools
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, GPT2LMHeadModel
from datasets import load_dataset

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
assert device.type == 'cuda', 'Switch Colab runtime to a T4 GPU before running.'

## 1. Config & seeds

`max_len`, `batch_size`, and the seeds match the class notebook so behaviour is comparable.

In [ ]:
# Reproducibility
torch.manual_seed(10); random.seed(10); np.random.seed(10)

# Hyperparameters (same spirit as Assignment 2)
max_len     = 100          # tokens per training example
batch_size  = 4
TRAIN_STEPS = 1000         # number of batches of continued pretraining

# Corpus settings
EDGAR_YEAR    = 'year_2020'   # which year of 10-K filings to stream
TARGET_CHUNKS = 15000         # how many ~120-word text chunks to build for the financial corpus
WORDS_PER_CHUNK = 120

## 2. Load GPT-2

We load GPT-2 once as the **base** model and make a deep copy that will receive the
additional financial pretraining, exactly as in class.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

base_gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
base_gpt2_model.resize_token_embeddings(len(tokenizer))   # account for the added [PAD] token

# The copy we will continue-pretrain on financial text:
additional_pretrain_model = copy.deepcopy(base_gpt2_model)
print('GPT-2 loaded. Vocab size:', len(tokenizer))

## 3. Helper classes & functions

These are the same building blocks from the class notebook:

- `ContinuedPretrainData` — tokenizes text to `max_len` and builds (input, next-token labels) pairs. It keeps only chunks long enough to fill `max_len`, so padding tokens never actually appear in the data.
- `ContinuedTrainingNetwork` — a thin wrapper that returns the GPT-2 next-token logits.
- `perplexity` — average `exp(cross-entropy)` over a loader: "how surprised is the model by the next token?"
- `continued_train_loop` — standard next-token-prediction training loop.

In [ ]:
class ContinuedPretrainData(Dataset):
    """Tokenize a list of strings into fixed-length next-token-prediction examples."""
    def __init__(self, base_data, tokenizer, max_len, device):
        self.max_len = max_len
        enc = tokenizer(base_data, max_length=max_len, truncation=True,
                        padding='max_length', return_tensors='pt')
        # keep only sequences that fill max_len (last position is a real token, not padding)
        keep = enc['attention_mask'][:, max_len - 1] > 0
        self.data = enc['input_ids'][keep].to(device)

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, i):
        return {'input':  self.data[i][:self.max_len - 1],   # tokens 0 .. n-2
                'labels': self.data[i][1:]}                  # tokens 1 .. n-1 (shifted)


def make_text_loader(texts, shuffle=False):
    """Wrap a list of strings into a DataLoader of (input, labels) pairs."""
    ds = ContinuedPretrainData(texts, tokenizer, max_len, device)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=True)


class ContinuedTrainingNetwork(nn.Module):
    def __init__(self, pretrain_model):
        super().__init__()
        self.pretrain_model = pretrain_model
    def forward(self, x):
        return self.pretrain_model(x).logits


loss_fn = nn.CrossEntropyLoss()


def perplexity(loader, model):
    losses = []
    for batch in loader:
        with torch.no_grad():
            try:
                out = model(batch['input'])
                out = out.reshape(batch_size * (max_len - 1), -1)
                lab = batch['labels'].reshape(batch_size * (max_len - 1))
                losses.append(loss_fn(out, lab).item())
            except Exception:
                continue
    return float(np.exp(np.mean(losses)))


def continued_train_loop(loader, model, loss_fn, optimizer, reporting_interval=100, steps=None):
    model.train()
    running, n = 0.0, 0
    for step, batch in enumerate(loader):
        if steps is not None and step >= steps:
            break
        out = model(batch['input']).reshape(-1, len(tokenizer))
        lab = batch['labels'].reshape(-1)
        optimizer.zero_grad()
        loss = loss_fn(out, lab)
        loss.backward()
        optimizer.step()
        running += loss.item(); n += 1
        if (step + 1) % reporting_interval == 0:
            avg = running / n
            print(f'  step {step+1:>4}: avg loss {avg:.4f} | perplexity {np.exp(avg):.2f}')
    avg = running / max(n, 1)
    print(f'Done. avg loss {avg:.4f} | perplexity {np.exp(avg):.2f}')

## 4. Build the in-domain (financial) corpus

We stream 10-K filings from `eloukas/edgar-corpus`, concatenate each filing's section text,
and split it into ~120-word chunks until we have `TARGET_CHUNKS`. This replaces the movie-plot
list from the class notebook.

In [ ]:
def chunk_text(text, words_per_chunk=WORDS_PER_CHUNK):
    w = text.split()
    return [' '.join(w[i:i+words_per_chunk]) for i in range(0, len(w), words_per_chunk)]

print('Streaming EDGAR 10-K filings...')
stream = load_dataset('eloukas/edgar-corpus', EDGAR_YEAR, split='train',
                      streaming=True, trust_remote_code=True)

edgar_chunks = []
for rec in stream:
    sections = [v for k, v in rec.items()
                if k.startswith('section_') and isinstance(v, str) and v.strip()]
    edgar_chunks.extend(chunk_text('\n'.join(sections)))
    if len(edgar_chunks) >= TARGET_CHUNKS:
        break

random.shuffle(edgar_chunks)
edgar_chunks = edgar_chunks[:TARGET_CHUNKS]
split = int(0.8 * len(edgar_chunks))
edgar_train_texts, edgar_test_texts = edgar_chunks[:split], edgar_chunks[split:]
print(f'EDGAR chunks: {len(edgar_chunks)}  (train {len(edgar_train_texts)} / test {len(edgar_test_texts)})')
print('Example chunk:\n', edgar_train_texts[0][:300], '...')

In [ ]:
edgar_train_loader = make_text_loader(edgar_train_texts, shuffle=True)
edgar_test_loader  = make_text_loader(edgar_test_texts,  shuffle=False)
print('Train batches:', len(edgar_train_loader), '| Test batches:', len(edgar_test_loader))

## 5. Build the out-of-domain corpus

General English from WikiText-2 — the control group. Continued pretraining on finance should
make the model *worse* (higher perplexity) here.

In [ ]:
wt = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test', trust_remote_code=True)
ood_text = ' '.join(t for t in wt['text'] if t.strip())
ood_chunks = chunk_text(ood_text)[:1500]
ood_loader = make_text_loader(ood_chunks, shuffle=False)
print('Out-of-domain (WikiText) batches:', len(ood_loader))

## 6. Perplexity BEFORE continued pretraining

Baseline numbers for in-domain (EDGAR) vs out-of-domain (WikiText) text, using the
untouched GPT-2.

In [ ]:
pretrain_network = ContinuedTrainingNetwork(additional_pretrain_model).to(device)

edgar_ppl_before = perplexity(edgar_test_loader, pretrain_network)
ood_ppl_before   = perplexity(ood_loader,        pretrain_network)
print(f'BEFORE  | EDGAR (in-domain):     {edgar_ppl_before:.2f}')
print(f'BEFORE  | WikiText (out-domain): {ood_ppl_before:.2f}')

## 7. Continued pretraining on financial text

Next-token prediction on the EDGAR training chunks for `TRAIN_STEPS` batches.

In [ ]:
optimizer = torch.optim.AdamW(pretrain_network.parameters(), lr=1e-5)
print(f'Continued pretraining for {TRAIN_STEPS} steps...')
continued_train_loop(edgar_train_loader, pretrain_network, loss_fn, optimizer, steps=TRAIN_STEPS)

## 8. Perplexity AFTER continued pretraining

In [ ]:
edgar_ppl_after = perplexity(edgar_test_loader, pretrain_network)
ood_ppl_after   = perplexity(ood_loader,        pretrain_network)

print('                         BEFORE     AFTER')
print(f'EDGAR (in-domain)      {edgar_ppl_before:8.2f}  {edgar_ppl_after:8.2f}   <- expect this to DROP')
print(f'WikiText (out-domain)  {ood_ppl_before:8.2f}  {ood_ppl_after:8.2f}   <- expect this to RISE')

## 9. What to look for

- **EDGAR perplexity went down** → the model got better at predicting financial-filing language. That's domain adaptation working.
- **WikiText perplexity went up (or held)** → the model traded some general-language ability for financial specialization. That trade-off is expected and is the whole point.

This is exactly the mechanism behind **FinBERT**: take a general model, continue pretraining on financial text, and you get a model that's better in-domain at the cost of some generality.

**Next steps (Parts 2–3, when you're ready):** fine-tune this financially-pretrained GPT-2 for sentiment on `takala/financial_phrasebank` and compare it to base GPT-2; then bring in BERT and compare `bert-base` vs `ProsusAI/finbert` to see the same lesson with a real published model.